# Week 5 · Day 3 — Linear Regression: The Basic Idea

This is our very first look at a machine-learning model. We keep it gentle: **no training algorithm yet, no calculus.** Today we just answer two questions:

1. **What *is* a linear regression model?** (Answer: a straight line.)
2. **How do we tell whether a line is good or bad?** (Answer: we measure its error.)

You'll draw lines through data by hand, try different lines yourself, and get a feel for what makes one line better than another. Next time, we'll let the computer find the best line automatically — but first you need to feel the problem with your own hands.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. The data

Machine learning starts with data: examples of an **input** and its matching **output**. Here the input is *hours studied* and the output is *marks obtained*. We have 5 students.

- We call the input **x** (the *feature*).
- We call the output **y** (the *target* — the thing we want to predict).

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)   # hours studied  (input, x)
y = np.array([3, 5, 6.5, 9, 10.5])           # marks obtained (output, y)

# How many examples do we have?
m = len(x)
print(f"We have m = {m} training examples:\n")
for i in range(m):
    print(f"  Student {i+1}: studied {x[i]:.0f} hours, scored {y[i]} marks")

### A note on notation
You'll see this everywhere in machine learning, so let's name it once:
- **m** = the number of examples (here, 5).
- **x⁽ⁱ⁾, y⁽ⁱ⁾** = the i-th example. So x⁽¹⁾ = 1 hour and y⁽¹⁾ = 3 marks.

*(The little ⁽¹⁾ means "example number 1" — it is **not** a power. x⁽²⁾ is the second student, not x-squared.)*

In [ ]:
# Look at one specific example — try changing i to 0, 1, 2, 3, or 4
i = 0
print(f"Example number {i+1}:  x = {x[i]} hours  ->  y = {y[i]} marks")

---
## 2. Always look at the data first

Before any modelling, plot the points. Each student is one dot.

In [ ]:
plt.scatter(x, y, marker="x", c="red", s=100)
plt.title("Study hours vs marks")
plt.xlabel("hours studied")
plt.ylabel("marks obtained")
plt.grid(True, alpha=0.3)
plt.show()

The dots clearly go **up from left to right** — more study, more marks. They almost fall along a straight line. That is the key observation: a straight line could describe this relationship well. Linear regression is simply the job of **finding that line.**

---
## 3. The model: a straight line

A linear regression model is just the equation of a straight line:

$$ f(x) = w \cdot x + b $$

You already know this from school as `y = mx + c`. Only the letters differ:

| School | Machine learning | Meaning |
|---|---|---|
| m (gradient) | **w** (weight) | the *slope* — how steep the line is |
| c | **b** (bias) | the *intercept* — where the line crosses the y-axis |

So a linear model is defined by just **two numbers: w and b.** Choose them, and you have a specific line. Our whole task is to choose *good* values.

In [ ]:
def f(x, w, b):
    """The model: a straight line. Given w and b, predict y for each x."""
    return w * x + b

Let's try a first guess. Suppose we pick `w = 1` and `b = 0`. That gives the line `f(x) = 1·x + 0`. Let's see what it predicts and draw it.

In [ ]:
w = 1
b = 0

predictions = f(x, w, b)
print(f"With w = {w} and b = {b}, the line is  f(x) = {w}x + {b}\n")
for i in range(m):
    print(f"  studied {x[i]:.0f}h -> model predicts {predictions[i]:.1f},  actual was {y[i]}")

In [ ]:
def plot_line(w, b, x, y, title=""):
    plt.scatter(x, y, marker="x", c="red", s=100, label="actual data", zorder=3)
    x_line = np.linspace(0, 6, 100)
    plt.plot(x_line, f(x_line, w, b), c="blue", label=f"f(x) = {w}x + {b}")
    plt.xlabel("hours studied"); plt.ylabel("marks")
    plt.title(title); plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 13)
    plt.show()

plot_line(w, b, x, y, title="First guess: w=1, b=0")

The blue line is our model; the red crosses are the real data. This line is **too shallow** — it sits below most of the points. Our guess of `w=1` isn't good. Let's try adjusting it.

---
## 4. Trying different lines by hand

The only way to improve is to change `w` and `b`. Let's build intuition by trying a few. 

**A steeper line** — increase the slope `w` to 3:

In [ ]:
plot_line(w=3, b=0, x=x, y=y, title="Steeper: w=3, b=0")

Now it's **too steep** — the line shoots up past the points. So `w=1` was too shallow and `w=3` is too steep. The good value is somewhere in between. Let's try `w=2`:

In [ ]:
plot_line(w=2, b=0, x=x, y=y, title="Better slope: w=2, b=0")

Much better! The slope looks right, but the whole line sits *slightly* below the points. We can lift it up by increasing the intercept `b`. Let's nudge `b` to 1:

In [ ]:
plot_line(w=2, b=1, x=x, y=y, title="Great fit: w=2, b=1")

That line runs right through the data. By hand, we found that **w ≈ 2, b ≈ 1** fits well. 

But notice what we just did — we *eyeballed* it. We said "too shallow," "too steep," "better," "great" by looking. That works for 5 points, but we need a **number** that tells us how good a line is, without relying on our eyes. That number is the *cost*.

---
## 5. Measuring how good a line is: the cost

For any line, each data point has an **error** — the gap between the real mark and the line's prediction:

$$ \text{error} = f(x) - y $$

A good line has small errors everywhere. To turn all those errors into a single score, we:

1. take each error,
2. **square** it (so a miss above and a miss below both count as positive, and big misses count more),
3. **average** the squared errors.

That single number is the **cost**. *Low cost = good line. High cost = bad line.*

$$ \text{cost} = \frac{1}{2m}\sum_{i=1}^{m}\big(f(x^{(i)}) - y^{(i)}\big)^2 $$

*(The extra ½ is just a convention that keeps later maths tidy — it doesn't change which line is best.)*

In [ ]:
def compute_cost(x, y, w, b):
    predictions = f(x, w, b)
    errors = predictions - y
    squared_errors = errors ** 2
    return np.sum(squared_errors) / (2 * len(x))

Let's look at the errors for our good line (w=2, b=1), one point at a time, so the formula feels concrete rather than abstract.

In [ ]:
w, b = 2, 1
print(f"Line: f(x) = {w}x + {b}\n")
print(f"{'hours':>6} {'actual':>8} {'predicted':>10} {'error':>8} {'error²':>8}")
for i in range(m):
    pred = f(x[i], w, b)
    err = pred - y[i]
    print(f"{x[i]:>6.0f} {y[i]:>8} {pred:>10.1f} {err:>8.1f} {err**2:>8.2f}")

print(f"\nCost for this line = {compute_cost(x, y, w, b):.3f}")

The errors are tiny, so the cost is tiny (about 0.05). That's the signature of a good line. Now let's confirm that **bad lines really do give bigger costs** — this is the whole point of the cost function.

In [ ]:
print("Cost for different lines:\n")
for w_try, b_try in [(1, 0), (3, 0), (2, 0), (2, 1)]:
    c = compute_cost(x, y, w_try, b_try)
    print(f"  w = {w_try}, b = {b_try}   ->   cost = {c:.3f}")

Read that table against what we saw by eye:
- `w=1, b=0` (too shallow) → cost **8.05** — high, a bad line.
- `w=3, b=0` (too steep) → cost **3.65** — still high.
- `w=2, b=0` (good slope, a bit low) → cost **0.30** — much better.
- `w=2, b=1` (our best) → cost **0.05** — lowest, the best line.

**The cost put a number on our intuition.** The line that looked best has the lowest cost. That's exactly what we want: a score that lets the *computer* judge lines without human eyes.

---
## 6. Seeing the cost change as we sweep the slope

Let's fix `b = 1` and try *many* values of `w`, computing the cost for each. Then we plot cost against `w`. This picture is worth a thousand words.

In [ ]:
b = 1
w_values = np.linspace(0, 4, 50)          # try 50 slopes from 0 to 4
costs = [compute_cost(x, y, w_try, b) for w_try in w_values]

plt.plot(w_values, costs, c="purple")
plt.scatter([2], [compute_cost(x, y, 2, b)], c="red", s=100, zorder=3, label="w=2 (best)")
plt.xlabel("w (slope)"); plt.ylabel("cost")
plt.title("Cost for different slopes (with b fixed at 1)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

This U-shaped curve is the heart of everything that comes next. Read it like this:
- Far to the left (small `w`, shallow line) → **high cost**.
- Far to the right (large `w`, steep line) → **high cost**.
- At the **bottom of the U** (around `w=2`) → **lowest cost** — the best slope.

Finding the best line means **finding the bottom of this curve.** Today we found it by trying values and looking. Next time, the computer will find the bottom automatically and efficiently — but it's chasing this exact same lowest point.

---
## 7. Using the model to predict

Once we have a good line, the payoff is **prediction** — estimating the mark for a number of study hours we've never seen. Let's use our best line (w=2, b=1) to predict the mark for a student who studies **3.5 hours**.

In [ ]:
w, b = 2, 1
hours = 3.5
predicted_marks = f(hours, w, b)
print(f"Our model:  f(x) = {w}x + {b}")
print(f"A student who studies {hours} hours is predicted to score {predicted_marks:.1f} marks")

And the slope has a real-world meaning: with `w = 2`, the model says **each extra hour of study is worth about 2 more marks.** That interpretability is one reason linear regression is so widely used — the model doesn't just predict, it *explains*.

---
## Summary — what you learned today

- A **linear regression model** is a straight line: `f(x) = w·x + b`. It's just `y = mx + c` with new names (**w** = slope, **b** = intercept).
- A model is defined by **two numbers, w and b**. Choosing them well is the whole game.
- The **cost** measures how good a line is: average the squared errors. **Low cost = good line.**
- Plotting cost against the slope gives a **U-shaped curve**; the best line sits at the bottom.
- A trained line lets you **predict** new values and even **explain** the relationship (each hour ≈ 2 marks).

Today you found a good line **by hand**, by trying values and watching the cost. That's fine for building intuition, but tedious and imprecise for real problems. 

**Next time:** we let the computer find the bottom of that U-curve automatically — the method is called *gradient descent*, and it's how every machine-learning model actually learns.

----

<div style="background:#0d1117;border:1px solid #21262d;border-radius:16px;padding:28px 30px;font-family:'Segoe UI',system-ui,sans-serif;box-shadow:0 6px 20px rgba(0,0,0,0.4);margin-top:15px;">
  <div style="display:flex;align-items:center;border-bottom:1px solid #21262d;padding-bottom:16px;margin-bottom:16px;">
    <img src="https://github.com/M-Abbas1.png" alt="profile"
         style="width:56px;height:56px;border-radius:50%;margin-right:16px;flex-shrink:0;border:2px solid #00c6a2;object-fit:cover;" />
    <div>
      <div style="color:#ffffff;font-size:20px;font-weight:700;">Muhammad Abbas</div>
      <div style="color:#00c6a2;font-size:14px;font-weight:500;">AI / Machine Learning Instructor</div>
    </div>
  </div>
  <div style="color:#8b949e;font-size:14px;line-height:1.9;">
    <div><span style="color:#00c6a2;">&#127891;</span>&nbsp; <b style="color:#c9d1d9;">Course:</b> Artificial Intelligence (Machine Learning &amp; Deep Learning)</div>
    <div><span style="color:#00c6a2;">&#127970;</span>&nbsp; <b style="color:#c9d1d9;">Program:</b> NAVTTC &mdash; Prime Minister&rsquo;s Hunarmand Pakistan &ldquo;Skills for All&rdquo;</div>
    <div style="margin-top:10px;display:flex;align-items:center;flex-wrap:wrap;gap:6px;">
      <img src="https://cdn.simpleicons.org/gmail/00c6a2?size=16" style="vertical-align:middle;" />
      <a href="mailto:mr.mabbas2@gmail.com" style="color:#58a6ff;text-decoration:none;">mr.mabbas2@gmail.com</a>
      &nbsp;&nbsp;|&nbsp;&nbsp;
      
      <a href="https://www.linkedin.com/in/muhammad-abbas-b93524279/" style="color:#58a6ff;text-decoration:none;">LinkedIn</a>
      &nbsp;&nbsp;|&nbsp;&nbsp;
      <img src="https://cdn.simpleicons.org/github/00c6a2?size=16" style="vertical-align:middle;" />
      <a href="https://github.com/M-Abbas1" style="color:#58a6ff;text-decoration:none;">GitHub</a>
    </div>
  </div>
  <div style="border-top:1px solid #21262d;margin-top:16px;padding-top:14px;text-align:center;">
    <span style="color:#6e7681;font-size:13px;font-style:italic;">
      &ldquo;The best way to predict the future is to create it.&rdquo;
    </span>
  </div>
  <div style="text-align:center;margin-top:10px;">
    <span style="color:#484f58;font-size:12px;">&copy; 2026 Muhammad Abbas &nbsp;&bull;&nbsp; Made with &#10084;&#65039; for learning</span>
  </div>
</div>